# Classical ML Notebook for DDoS Detection

This notebook solves the **QCentroid × GSMA DDoS anomaly detection challenge** with a fully classical pipeline.

The design is intentionally **quantum-inspired** rather than quantum-native:
- it maps aggregated network-flow features into a higher-dimensional nonlinear space with **Random Fourier Features** via `RBFSampler`;
- this approximates an RBF kernel feature map, which is conceptually similar to using a rich feature embedding as in kernelized or quantum-kernel methods;
- the downstream classifier is a simple, transparent **logistic regression** model.

The pipeline predicts:
1. whether each dataset contains DDoS activity;
2. which `src_ip` values are responsible for the anomalous behaviour.

## Approach

The notebook works at the **source-IP level** instead of the raw-flow level. For each dataset, it aggregates all flows belonging to the same `src_ip` and builds features such as:

- flow volume and concentration;
- concentration toward a small set of `dst_ip` / `dst_port`;
- packet and byte rates;
- packet-size behaviour;
- imbalance in `outbound_byte_ratio`;
- protocol mix.

This is a good fit for DDoS because attacks usually appear as many aggressive flows from the same source set toward a narrow target set.

We train **one model per family** because the two families come from different attack-generation processes. Dataset-level detection is then derived from the best source-IP score inside each dataset.

In [7]:
from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.kernel_approximation import RBFSampler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, precision_recall_fscore_support, roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 160)
np.random.seed(42)

In [8]:
ROOT = Path.cwd()

FAMILY_PATHS = {
    "family_a": ROOT / "Datasets" / "Datasets" / "Option 1" / "option1_nf_unsw_dos_as_ddos_reduced_schema",
    "family_b": ROOT / "Datasets" /"Datasets" / "Option 2" / "option2_nf_unsw_base_cse_native_ddos_reduced_schema",
}

SOURCE_NUMERIC_COLUMNS = [
    "flow_count",
    "unique_dst_ip",
    "unique_dst_port",
    "unique_protocols",
    "pps_mean",
    "pps_max",
    "bps_mean",
    "bps_max",
    "duration_mean",
    "duration_max",
    "packet_size_avg_mean",
    "packet_size_avg_std",
    "outbound_ratio_mean",
    "outbound_ratio_min",
    "inter_arrival_mean_mean",
    "inter_arrival_std_mean",
    "total_packets_sum",
    "total_bytes_sum",
    "dst_concentration",
    "dst_port_concentration",
    "tcp_share",
    "udp_share",
    "icmp_share",
    "small_packet_share",
    "high_pps_share",
    "low_outbound_share",
]

SOURCE_CATEGORICAL_COLUMNS = ["top_dst_ip", "top_protocol"]
SPLITS = ["train", "validation", "test"]
SCENARIOS = ["normal", "attack"]

In [9]:
def build_catalog() -> pd.DataFrame:
    rows = []
    for family_name, family_path in FAMILY_PATHS.items():
        for scenario in SCENARIOS:
            for split in SPLITS:
                folder = family_path / scenario / split
                for csv_path in sorted(folder.glob("*.csv")):
                    rows.append(
                        {
                            "family": family_name,
                            "scenario": scenario,
                            "split": split,
                            "dataset_name": csv_path.stem,
                            "dataset_path": csv_path,
                        }
                    )
    return pd.DataFrame(rows).sort_values(["family", "split", "scenario", "dataset_name"]).reset_index(drop=True)


catalog = build_catalog()
catalog.head(10)

,family,scenario,split,dataset_name,dataset_path
0,family_a,attack,test,attack_test_01,c:\Users\bingg\OneDrive\Uni\Obsidian Notes\FS ...
1,family_a,attack,test,attack_test_02,c:\Users\bingg\OneDrive\Uni\Obsidian Notes\FS ...
2,family_a,normal,test,normal_test_01,c:\Users\bingg\OneDrive\Uni\Obsidian Notes\FS ...
3,family_a,normal,test,normal_test_02,c:\Users\bingg\OneDrive\Uni\Obsidian Notes\FS ...
4,family_a,attack,train,attack_train_01,c:\Users\bingg\OneDrive\Uni\Obsidian Notes\FS ...
5,family_a,attack,train,attack_train_02,c:\Users\bingg\OneDrive\Uni\Obsidian Notes\FS ...
6,family_a,attack,train,attack_train_03,c:\Users\bingg\OneDrive\Uni\Obsidian Notes\FS ...
7,family_a,attack,train,attack_train_04,c:\Users\bingg\OneDrive\Uni\Obsidian Notes\FS ...
8,family_a,attack,train,attack_train_05,c:\Users\bingg\OneDrive\Uni\Obsidian Notes\FS ...
9,family_a,attack,train,attack_train_06,c:\Users\bingg\OneDrive\Uni\Obsidian Notes\FS ...


In [10]:
catalog.groupby(["family", "split", "scenario"]).size().rename("datasets").reset_index()

,family,split,scenario,datasets
0,family_a,test,attack,2
1,family_a,test,normal,2
2,family_a,train,attack,8
3,family_a,train,normal,8
4,family_a,validation,attack,2
5,family_a,validation,normal,2
6,family_b,test,attack,2
7,family_b,test,normal,2
8,family_b,train,attack,8
9,family_b,train,normal,8


In [11]:
def load_dataset(csv_path: Path) -> pd.DataFrame:
    df = pd.read_csv(csv_path, low_memory=False)
    df["protocol"] = df["protocol"].astype(str)
    return df


sample_df = load_dataset(catalog.iloc[0]["dataset_path"])
sample_df.head(3)

,src_ip,dst_ip,src_port,dst_port,protocol,outbound_byte_ratio,duration,packets_per_second,bytes_per_second,inter_packet_arrival_mean,inter_packet_arrival_std,total_packets,total_bytes,packet_size_avg,packet_size_std,Label,Attack,scenario,split,dataset_id,row_in_window,is_seeded_ddos,burst_id,burst_phase,source_dataset
0,59.166.0.7,149.171.126.9,57060,29409,TCP,0.941733,0.098,1734.693878,869673.469388,0.000000,1.000000,170,85228,501.341176,559.085369,0,Benign,attack,test,1,0,0,NaN,NaN,NF-UNSW-NB15-v3-Benign
1,59.166.0.8,149.171.126.0,20605,21,TCP,0.955446,0.001,4000.000000,202000.000000,0.000000,0.000000,4,202,50.500000,0.000000,0,Benign,attack,test,1,1,0,NaN,NaN,NF-UNSW-NB15-v3-Benign
2,59.166.0.0,149.171.126.6,20513,23156,TCP,0.852806,0.046,304.347826,47260.869565,7.714286,16.845638,14,2174,155.285714,246.348683,0,Benign,attack,test,1,2,0,NaN,NaN,NF-UNSW-NB15-v3-Benign


In [12]:
def source_feature_table(df: pd.DataFrame, family: str, split: str, scenario: str, dataset_name: str) -> pd.DataFrame:
    work = df.copy()
    work["small_packet"] = (work["packet_size_avg"] <= 200).astype(int)
    work["low_outbound"] = (work["outbound_byte_ratio"] <= 0.20).astype(int)
    pps_threshold = work["packets_per_second"].quantile(0.95)
    work["high_pps"] = (work["packets_per_second"] >= pps_threshold).astype(int)

    rows = []
    for src_ip, group in work.groupby("src_ip", sort=False):
        dst_counts = group["dst_ip"].value_counts(dropna=False)
        dst_port_counts = group["dst_port"].astype(str).value_counts(dropna=False)
        protocol_counts = group["protocol"].value_counts(dropna=False)

        rows.append(
            {
                "family": family,
                "split": split,
                "scenario": scenario,
                "dataset_name": dataset_name,
                "src_ip": src_ip,
                "flow_count": len(group),
                "unique_dst_ip": group["dst_ip"].nunique(),
                "unique_dst_port": group["dst_port"].nunique(),
                "unique_protocols": group["protocol"].nunique(),
                "pps_mean": group["packets_per_second"].mean(),
                "pps_max": group["packets_per_second"].max(),
                "bps_mean": group["bytes_per_second"].mean(),
                "bps_max": group["bytes_per_second"].max(),
                "duration_mean": group["duration"].mean(),
                "duration_max": group["duration"].max(),
                "packet_size_avg_mean": group["packet_size_avg"].mean(),
                "packet_size_avg_std": group["packet_size_avg"].std(ddof=0),
                "outbound_ratio_mean": group["outbound_byte_ratio"].mean(),
                "outbound_ratio_min": group["outbound_byte_ratio"].min(),
                "inter_arrival_mean_mean": group["inter_packet_arrival_mean"].mean(),
                "inter_arrival_std_mean": group["inter_packet_arrival_std"].mean(),
                "total_packets_sum": group["total_packets"].sum(),
                "total_bytes_sum": group["total_bytes"].sum(),
                "dst_concentration": dst_counts.iloc[0] / len(group),
                "dst_port_concentration": dst_port_counts.iloc[0] / len(group),
                "top_dst_ip": str(dst_counts.index[0]),
                "top_protocol": str(protocol_counts.index[0]),
                "tcp_share": (group["protocol"] == "TCP").mean(),
                "udp_share": (group["protocol"] == "UDP").mean(),
                "icmp_share": (group["protocol"] == "ICMP").mean(),
                "small_packet_share": group["small_packet"].mean(),
                "high_pps_share": group["high_pps"].mean(),
                "low_outbound_share": group["low_outbound"].mean(),
                "is_malicious_source": int(group["is_seeded_ddos"].max()),
            }
        )

    source_df = pd.DataFrame(rows)
    source_df["packet_size_avg_std"] = source_df["packet_size_avg_std"].fillna(0.0)
    return source_df

In [13]:
source_tables = []
for row in catalog.itertuples(index=False):
    df = load_dataset(row.dataset_path)
    source_tables.append(
        source_feature_table(
            df=df,
            family=row.family,
            split=row.split,
            scenario=row.scenario,
            dataset_name=row.dataset_name,
        )
    )

source_data = pd.concat(source_tables, ignore_index=True)
source_data.head()

,family,split,scenario,dataset_name,src_ip,flow_count,unique_dst_ip,unique_dst_port,unique_protocols,pps_mean,pps_max,bps_mean,bps_max,duration_mean,duration_max,packet_size_avg_mean,packet_size_avg_std,outbound_ratio_mean,outbound_ratio_min,inter_arrival_mean_mean,inter_arrival_std_mean,total_packets_sum,total_bytes_sum,dst_concentration,dst_port_concentration,top_dst_ip,top_protocol,tcp_share,udp_share,icmp_share,small_packet_share,high_pps_share,low_outbound_share,is_malicious_source
0,family_a,test,attack,attack_test_01,59.166.0.7,9360,10,2934,2,2822.727850,15000.000000,525052.617439,3.117333e+06,0.491479,51.314,219.688212,193.897715,0.649811,0.0,14.614867,39.369458,776915,375459906,0.106410,0.211325,149.171.126.0,TCP,0.772650,0.227350,0.0,0.618483,0.064530,0.052350,0
1,family_a,test,attack,attack_test_01,59.166.0.8,9390,10,2878,2,2807.146043,14705.882353,525163.072909,4.572000e+06,0.466170,42.487,218.364047,192.026315,0.651300,0.0,14.514838,39.120574,772757,362940495,0.105857,0.220980,149.171.126.1,TCP,0.781470,0.218530,0.0,0.618637,0.063685,0.051757,0
2,family_a,test,attack,attack_test_01,59.166.0.0,9797,10,2919,2,2793.548094,15333.333333,517377.692440,3.117333e+06,0.470570,45.287,218.259197,193.954337,0.646934,0.0,14.201875,38.766856,809708,393494328,0.109523,0.217618,149.171.126.0,TCP,0.770644,0.229356,0.0,0.621823,0.060937,0.054098,0
3,family_a,test,attack,attack_test_01,59.166.0.4,9782,10,3046,2,2857.151556,13888.888889,535781.963064,4.572000e+06,0.464926,41.314,222.322548,196.290635,0.649103,0.0,13.072322,35.774697,840674,414947073,0.107544,0.208444,149.171.126.4,TCP,0.773359,0.226641,0.0,0.615007,0.064200,0.054386,0
4,family_a,test,attack,attack_test_01,59.166.0.1,9982,10,2962,2,2734.413435,15333.333333,511894.821941,3.117333e+06,0.522160,49.039,219.241150,195.215006,0.646690,0.0,15.121019,39.882079,851229,418298251,0.105891,0.219495,149.171.126.0,TCP,0.778000,0.222000,0.0,0.621419,0.059507,0.054498,0


In [14]:
source_data.groupby(["family", "split", "scenario"])["src_ip"].count().rename("source_rows").reset_index()

,family,split,scenario,source_rows
0,family_a,test,attack,229
1,family_a,test,normal,67
2,family_a,train,attack,804
3,family_a,train,normal,272
4,family_a,validation,attack,210
5,family_a,validation,normal,68
6,family_b,test,attack,86
7,family_b,test,normal,67
8,family_b,train,attack,354
9,family_b,train,normal,271


In [15]:
@dataclass
class FamilyArtifacts:
    family: str
    model: Pipeline
    source_threshold: float
    dataset_threshold: float


def build_model() -> Pipeline:
    preprocess = ColumnTransformer(
        transformers=[
            (
                "num",
                Pipeline(
                    steps=[
                        ("scale", StandardScaler()),
                        ("rff", RBFSampler(gamma=0.5, n_components=256, random_state=42)),
                    ]
                ),
                SOURCE_NUMERIC_COLUMNS,
            ),
            ("cat", OneHotEncoder(handle_unknown="ignore"), SOURCE_CATEGORICAL_COLUMNS),
        ],
        remainder="drop",
    )

    return Pipeline(
        steps=[
            ("preprocess", preprocess),
            ("classifier", LogisticRegression(max_iter=2000, class_weight="balanced")),
        ]
    )


def dataset_prediction_frame(source_predictions: pd.DataFrame, threshold: float) -> pd.DataFrame:
    work = source_predictions.copy()
    work["predicted_source"] = (work["malicious_probability"] >= threshold).astype(int)
    return (
        work.groupby(["family", "split", "scenario", "dataset_name"], as_index=False)
        .agg(
            true_dataset_label=("is_malicious_source", "max"),
            max_source_probability=("malicious_probability", "max"),
            mean_source_probability=("malicious_probability", "mean"),
            predicted_source_count=("predicted_source", "sum"),
        )
    )


def fit_family_model(family_name: str, family_source_data: pd.DataFrame) -> FamilyArtifacts:
    train_df = family_source_data[family_source_data["split"] == "train"].copy()

    model = build_model()
    model.fit(train_df[SOURCE_NUMERIC_COLUMNS + SOURCE_CATEGORICAL_COLUMNS], train_df["is_malicious_source"])

    return FamilyArtifacts(
        family=family_name,
        model=model,
        source_threshold=0.5,
        dataset_threshold=0.5,
    )

In [16]:
family_artifacts = {}
for family_name, family_df in source_data.groupby("family"):
    family_artifacts[family_name] = fit_family_model(family_name, family_df)

{
    family: {
        "source_threshold": artifacts.source_threshold,
        "dataset_threshold": artifacts.dataset_threshold,
    }
    for family, artifacts in family_artifacts.items()
}

{'family_a': {'source_threshold': 0.5, 'dataset_threshold': 0.5},
 'family_b': {'source_threshold': 0.5, 'dataset_threshold': 0.5}}

In [17]:
def score_family_split(family_name: str, split_name: str, family_df: pd.DataFrame, artifacts: FamilyArtifacts):
    split_df = family_df[family_df["split"] == split_name].copy()
    split_df["malicious_probability"] = artifacts.model.predict_proba(
        split_df[SOURCE_NUMERIC_COLUMNS + SOURCE_CATEGORICAL_COLUMNS]
    )[:, 1]
    split_df["predicted_source"] = (split_df["malicious_probability"] >= artifacts.source_threshold).astype(int)

    dataset_df = dataset_prediction_frame(split_df, artifacts.source_threshold)
    dataset_df["predicted_dataset_label"] = (
        dataset_df["max_source_probability"] >= artifacts.dataset_threshold
    ).astype(int)

    source_precision, source_recall, source_f1, _ = precision_recall_fscore_support(
        split_df["is_malicious_source"],
        split_df["predicted_source"],
        average="binary",
        zero_division=0,
    )
    dataset_precision, dataset_recall, dataset_f1, _ = precision_recall_fscore_support(
        dataset_df["true_dataset_label"],
        dataset_df["predicted_dataset_label"],
        average="binary",
        zero_division=0,
    )

    metrics = pd.DataFrame(
        [
            {
                "family": family_name,
                "split": split_name,
                "level": "source_ip",
                "precision": source_precision,
                "recall": source_recall,
                "f1": source_f1,
                "auc": roc_auc_score(split_df["is_malicious_source"], split_df["malicious_probability"]),
            },
            {
                "family": family_name,
                "split": split_name,
                "level": "dataset",
                "precision": dataset_precision,
                "recall": dataset_recall,
                "f1": dataset_f1,
                "auc": roc_auc_score(dataset_df["true_dataset_label"], dataset_df["max_source_probability"]),
            },
        ]
    )
    return split_df, dataset_df, metrics

In [18]:
metric_frames = []
for family_name, family_df in source_data.groupby("family"):
    artifacts = family_artifacts[family_name]
    for split_name in ["validation", "test"]:
        _, _, metrics_df = score_family_split(
            family_name=family_name,
            split_name=split_name,
            family_df=family_df,
            artifacts=artifacts,
        )
        metric_frames.append(metrics_df)

metrics_table = pd.concat(metric_frames, ignore_index=True)
metrics_table

,family,split,level,precision,recall,f1,auc
0,family_a,validation,source_ip,1.0,1.0,1.0,1.0
1,family_a,validation,dataset,1.0,1.0,1.0,1.0
2,family_a,test,source_ip,1.0,1.0,1.0,1.0
3,family_a,test,dataset,1.0,1.0,1.0,1.0
4,family_b,validation,source_ip,1.0,1.0,1.0,1.0
5,family_b,validation,dataset,1.0,1.0,1.0,1.0
6,family_b,test,source_ip,1.0,1.0,1.0,1.0
7,family_b,test,dataset,1.0,1.0,1.0,1.0


In [19]:
for family_name, family_df in source_data.groupby("family"):
    artifacts = family_artifacts[family_name]
    validation_df = family_df[family_df["split"] == "validation"].copy()
    validation_df["malicious_probability"] = artifacts.model.predict_proba(
        validation_df[SOURCE_NUMERIC_COLUMNS + SOURCE_CATEGORICAL_COLUMNS]
    )[:, 1]
    validation_df["predicted_source"] = (validation_df["malicious_probability"] >= artifacts.source_threshold).astype(int)
    print(f"\n=== {family_name} | source-IP validation report ===")
    print(classification_report(validation_df["is_malicious_source"], validation_df["predicted_source"], digits=4))


=== family_a | source-IP validation report ===
              precision    recall  f1-score   support

           0     1.0000    1.0000    1.0000       134
           1     1.0000    1.0000    1.0000       144

    accuracy                         1.0000       278
   macro avg     1.0000    1.0000    1.0000       278
weighted avg     1.0000    1.0000    1.0000       278


=== family_b | source-IP validation report ===
              precision    recall  f1-score   support

           0     1.0000    1.0000    1.0000       140
           1     1.0000    1.0000    1.0000        20

    accuracy                         1.0000       160
   macro avg     1.0000    1.0000    1.0000       160
weighted avg     1.0000    1.0000    1.0000       160



In [20]:
all_source_predictions = []
all_dataset_predictions = []

for family_name, family_df in source_data.groupby("family"):
    artifacts = family_artifacts[family_name]
    scored = family_df.copy()
    scored["malicious_probability"] = artifacts.model.predict_proba(
        scored[SOURCE_NUMERIC_COLUMNS + SOURCE_CATEGORICAL_COLUMNS]
    )[:, 1]
    scored["predicted_source"] = (scored["malicious_probability"] >= artifacts.source_threshold).astype(int)
    all_source_predictions.append(scored)

    dataset_df = dataset_prediction_frame(scored, artifacts.source_threshold)
    dataset_df["predicted_dataset_label"] = (
        dataset_df["max_source_probability"] >= artifacts.dataset_threshold
    ).astype(int)
    all_dataset_predictions.append(dataset_df)

all_source_predictions = pd.concat(all_source_predictions, ignore_index=True)
all_dataset_predictions = pd.concat(all_dataset_predictions, ignore_index=True)

In [21]:
shortlist_df = (
    all_source_predictions[all_source_predictions["predicted_source"] == 1]
    .sort_values(["family", "split", "dataset_name", "malicious_probability"], ascending=[True, True, True, False])
    .groupby(["family", "split", "scenario", "dataset_name"], as_index=False)
    .agg(
        predicted_source_ips=("src_ip", lambda x: ";".join(map(str, x))),
        shortlisted_source_count=("src_ip", "count"),
    )
)
shortlist_df.head()

,family,split,scenario,dataset_name,predicted_source_ips,shortlisted_source_count
0,family_a,test,attack,attack_test_01,10.46.195.235;10.244.108.70;10.43.75.186;10.31...,89
1,family_a,test,attack,attack_test_02,10.14.8.78;10.11.73.41;10.101.104.90;10.146.58...,71
2,family_a,train,attack,attack_train_01,10.22.83.214;10.184.166.178;10.37.39.9;10.221....,59
3,family_a,train,attack,attack_train_02,10.185.147.26;10.59.46.74;10.68.6.159;10.194.1...,71
4,family_a,train,attack,attack_train_03,10.14.74.245;10.132.255.139;10.58.164.90;10.11...,69


In [22]:
final_predictions = all_dataset_predictions.merge(
    shortlist_df,
    on=["family", "split", "scenario", "dataset_name"],
    how="left",
)

final_predictions["predicted_source_ips"] = final_predictions["predicted_source_ips"].fillna("")
final_predictions["shortlisted_source_count"] = final_predictions["shortlisted_source_count"].fillna(0).astype(int)
final_predictions = final_predictions.sort_values(["family", "split", "scenario", "dataset_name"]).reset_index(drop=True)
final_predictions.head(12)

,family,split,scenario,dataset_name,true_dataset_label,max_source_probability,mean_source_probability,predicted_source_count,predicted_dataset_label,predicted_source_ips,shortlisted_source_count
0,family_a,test,attack,attack_test_01,1,0.995940,0.700800,89,1,10.46.195.235;10.244.108.70;10.43.75.186;10.31...,89
1,family_a,test,attack,attack_test_02,1,0.995393,0.655991,71,1,10.14.8.78;10.11.73.41;10.101.104.90;10.146.58...,71
2,family_a,test,normal,normal_test_01,0,0.285769,0.050509,0,0,,0
3,family_a,test,normal,normal_test_02,0,0.223064,0.041647,0,0,,0
4,family_a,train,attack,attack_train_01,1,0.996640,0.629624,59,1,10.22.83.214;10.184.166.178;10.37.39.9;10.221....,59
5,family_a,train,attack,attack_train_02,1,0.995309,0.655181,71,1,10.185.147.26;10.59.46.74;10.68.6.159;10.194.1...,71
6,family_a,train,attack,attack_train_03,1,0.995370,0.657491,69,1,10.14.74.245;10.132.255.139;10.58.164.90;10.11...,69
7,family_a,train,attack,attack_train_04,1,0.995429,0.623973,58,1,10.193.24.125;10.100.22.103;10.179.61.14;10.14...,58
8,family_a,train,attack,attack_train_05,1,0.996708,0.677129,74,1,10.150.167.164;10.77.154.8;10.213.61.85;10.46....,74
9,family_a,train,attack,attack_train_06,1,0.996076,0.657053,74,1,10.250.159.192;10.226.109.198;10.54.167.11;10....,74


In [23]:
output_dir = ROOT / "outputs"
output_dir.mkdir(exist_ok=True)

full_output_path = output_dir / "classical_qi_predictions_all_splits.csv"
test_output_path = output_dir / "classical_qi_predictions_test_only.csv"

final_predictions.to_csv(full_output_path, index=False)
final_predictions[final_predictions["split"] == "test"].to_csv(test_output_path, index=False)

print(full_output_path)
print(test_output_path)

c:\Users\bingg\OneDrive\Uni\Obsidian Notes\FS 2026\QuString\outputs\classical_qi_predictions_all_splits.csv
c:\Users\bingg\OneDrive\Uni\Obsidian Notes\FS 2026\QuString\outputs\classical_qi_predictions_test_only.csv


## Notes

- The model never uses audit columns such as `scenario`, `split`, `dataset_id`, `burst_id` or `is_seeded_ddos` as input features.
- The labels are only used for supervised training and offline evaluation.
- The exported CSV uses one row per dataset and includes both the dataset-level DDoS flag and a semicolon-separated source-IP shortlist.
- The thresholds are fixed at `0.5`, which keeps the decision rule simple and stable across both families.
- If the hackathon starter notebook later enforces a slightly different submission schema, only the final export cell should need a small edit.